# EDA — Superstore Canada (SQLite)


**Objectif :** analyser les ventes et la rentabilité (profit/marge) afin d’identifier les tendances, les segments/catégories/régions clés, et l’impact des remises.

**Questions business :**
- Comment évoluent ventes & profit dans le temps ?
- Quelles catégories génèrent le plus de profit ? lesquelles détruisent de la marge ?
- Quel est l’impact des remises (discount) sur le profit ?
- Quelles régions / modes de livraison performent le mieux ?

## 1.Connexion aux bases de données

On se connecte à deux bases :
- RAW : données brutes
- CLEAN : données nettoyées et normalisées

Cette séparation permet de comparer la qualité des données et travailler sur une version fiable.

In [ ]:
# Connexion aux bases de données( RAW et CLEAN )
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

RAW_DB   = "../data/superstore.db"
CLEAN_DB = "../data/superstore_clean.db"

print("RAW exists:", Path(RAW_DB).exists())
print("CLEAN exists:", Path(CLEAN_DB).exists())

con_raw = sqlite3.connect(RAW_DB)
con_raw.execute("PRAGMA foreign_keys = ON;")

con_clean = sqlite3.connect(CLEAN_DB)
con_clean.execute("PRAGMA foreign_keys = ON;")

print(" connecté à RAW:", RAW_DB)
print(" connecté à CLEAN:", CLEAN_DB)

RAW exists: True
CLEAN exists: True
✅ connecté à RAW: ../data/superstore.db
✅ connecté à CLEAN: ../data/superstore_clean.db


In [ ]:
# Exploration de la structure de la base

tables = pd.read_sql_query("""
SELECT name, type
FROM sqlite_master
WHERE type IN ('table','view')
ORDER BY type, name;
""", con_raw)
tables

,name,type
0,customers,table
1,dim_category,table
2,dim_date,table
3,dim_geo,table
4,dim_order_priority,table
5,dim_segment,table
6,dim_ship_mode,table
7,order_items,table
8,orders,table
9,products,table


## 2. Vérification d’intégrité : relation commandes → items

On vérifie s’il existe des items sans commande correspondante.
Résultat attendu : 0 (sinon problème d’intégrité).

In [ ]:
# order_items qui n'ont pas de commande correspondant(devrait etre 0)
q1 = """
SELECT COUNT(*) AS orphan_items
FROM order_items oi
LEFT JOIN orders o ON o.Order_ID = oi.Order_ID
WHERE o.Order_ID IS NULL;
"""
pd.read_sql_query(q1, con_clean)

,orphan_items
0,0


## 3. Vérification d’intégrité : relation produits → items

On vérifie si certains items ne sont liés à aucun produit.
Résultat attendu : 0.

In [ ]:
# order_items qui n'ont pas de produit correspondant (devrait être 0)
q2 = """
SELECT COUNT(*) AS orphan_items
FROM order_items oi
LEFT JOIN products p ON p.Product_ID = oi.Product_ID
WHERE p.Product_ID IS NULL;
"""
pd.read_sql_query(q2, con_clean)

,orphan_items
0,0


## 4. Fonction utilitaire(list_objects) : inspection des bases

Cette fonction permet d’afficher les tables et vues d’une base donnée.
On compare ensuite RAW vs CLEAN.

In [ ]:
import sqlite3
import pandas as pd

def list_objects(db_path):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query("""
        SELECT type, name
        FROM sqlite_master
        WHERE type IN ('table','view')
        ORDER BY type, name;
    """, con)
    con.close()
    return df

print("RAW:")
display(list_objects("../data/superstore.db"))

print("CLEAN:")
display(list_objects("../data/superstore_clean.db"))

RAW:


,type,name
0,table,customers
1,table,dim_category
2,table,dim_date
3,table,dim_geo
4,table,dim_order_priority
5,table,dim_segment
6,table,dim_ship_mode
7,table,order_items
8,table,orders
9,table,products


CLEAN:


,type,name
0,table,customers
1,table,dim_category
2,table,dim_date
3,table,dim_geo
4,table,dim_order_priority
5,table,dim_segment
6,table,dim_ship_mode
7,table,order_items
8,table,orders
9,table,products


## 5. Analyse de la rentabilité

On calcule :
- le nombre total d’items
- le nombre d’items avec profit négatif
- le pourcentage de pertes

Objectif : mesurer le problème de rentabilité.

In [ ]:
# Profits négatifs
pd.read_sql_query("""
SELECT
  COUNT(*) AS n_items,
  SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) AS n_neg_profit,
  ROUND(100.0 * SUM(CASE WHEN Profit < 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_neg_profit
FROM order_items;
""", con_clean)

,n_items,n_neg_profit,pct_neg_profit
0,51290,12544,24.46


## 6. Analyse des remises élevées

On mesure la proportion de transactions avec une remise > 30%.

Objectif : comprendre si les promotions peuvent expliquer les pertes.

In [ ]:
# Remises élevées (ex: > 30%)
pd.read_sql_query("""
SELECT
  SUM(CASE WHEN Discount > 0.30 THEN 1 ELSE 0 END) AS n_gt_30,
  ROUND(100.0 * SUM(CASE WHEN Discount > 0.30 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_gt_30
FROM order_items
WHERE Discount IS NOT NULL;
""", con_clean)

,n_gt_30,pct_gt_30
0,10361,20.2


## 7. Analyse des commandes multiples

On identifie les Order_ID apparaissant plusieurs fois.

Important : ce n’est pas forcément une erreur (une commande peut contenir plusieurs items).

In [ ]:
# ) Doublons sur Order_ID dans sales_raw
pd.read_sql_query("""
SELECT Order_ID, COUNT(*) AS n
FROM sales_raw
GROUP BY Order_ID
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 10;
""", con_raw)

,Order_ID,n
0,CA-2014-100111,14
1,TO-2014-9950,13
2,NI-2014-8880,13
3,MX-2014-166541,13
4,IN-2013-42311,13
5,IN-2012-41261,13
6,MX-2013-142678,12
7,MX-2013-127705,12
8,IN-2014-15263,12
9,IN-2011-76625,12


## Interprétation des doublons
Avoir plusieurs lignes par Order_ID est normal :
une commande peut contenir plusieurs produits.
Il faut interpréter les résultats, pas conclure automatiquement à une erreur.

## 8. Détection des valeurs extrêmes (outliers)

On analyse :
- les ventes (Sales)
- les coûts de livraison (Shipping_Cost)

Objectif : identifier des transactions atypiques pouvant biaiser l’analyse.

In [ ]:
# Valeurs extrêmes (outliers) sur Sales / Shipping_Cost
pd.read_sql_query("""
SELECT
  MIN(Sales) AS min_sales,
  MAX(Sales) AS max_sales,
  MIN(Shipping_Cost) AS min_ship,
  MAX(Shipping_Cost) AS max_ship
FROM sales_raw;
""", con_raw)

,min_sales,max_sales,min_ship,max_ship
0,0.444,22638.48,0.0,933.57


## 9. Vérification des identifiants uniques

On vérifie si Row_ID est bien unique.

Résultat : présence de doublons → problème potentiel de qualité des données.

In [ ]:
# vérifier Row_ID répétées( row_ID censé etre unique)
pd.read_sql_query("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT Row_ID) AS distinct_row_id,
  COUNT(*) - COUNT(DISTINCT Row_ID) AS duplicated_row_id
FROM sales_raw;
""", con_raw)

,total_rows,distinct_row_id,duplicated_row_id
0,51292,51290,2


## 10. Analyse des lignes corrompues

On extrait les lignes avec Row_ID NULL.

Résultat : 2 lignes totalement vides → données corrompues à ignorer.

In [ ]:
pd.read_sql_query("""
SELECT *
FROM sales_raw
WHERE Row_ID IS NULL;
""", con_raw)

,Row_ID,Order_ID,Order_Date,Ship_Date,Number_of_days,Ship_Mode,Customer_ID,Customer_Name,Segment,City,...,Sales,Quantity,Unit_Sales,Discount,Profit,Shipping_Cost,Order_Priority,Unit_shipping_cost,Profit_per_unit,unit_cost
0,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
# Fermeture des connexions
con_raw.close()
con_clean.close()